In [1]:
import os # for handling file paths
import pandas as pd  # data manipulation
import holidays # for handling holidays
from astral.sun import dawn, dusk  # for calculating sunrise/sunset times
from astral import LocationInfo # for defining location for sunrise/sunset calculations

In [2]:
cwd = os.getcwd()

parent_dir = os.path.dirname(cwd)
residential4_hourly_data_path = os.path.join(parent_dir, 'data/raw/residential4_hourly_merged.csv')
residential4_daily_data_path = os.path.join(parent_dir, 'data/raw/residential4_daily_merged.csv')

residential4_hourly_data = pd.read_csv(residential4_hourly_data_path, index_col=0, parse_dates=True)
residential4_daily_data = pd.read_csv(residential4_daily_data_path, index_col=0, parse_dates=True)

residential4_hourly_data['energy_demand'] = residential4_hourly_data['grid_import']
residential4_daily_data['energy_demand'] = residential4_daily_data['grid_import']

residential4_hourly_data = residential4_hourly_data.drop(columns=['grid_import'])
residential4_daily_data = residential4_daily_data.drop(columns=['grid_import'])

residential4_hourly_data.head(3)

,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,energy_demand
utc_timestamp,,,,,,,,,,,
2015-10-13 17:00:00+00:00,0.002,0.57,0.018,0.0,0.47,0.0,0.002,5.224,0.0,0.0,1.248
2015-10-13 18:00:00+00:00,0.000,0.00,0.026,0.0,0.53,0.0,0.000,4.540,0.0,0.0,0.765
2015-10-13 19:00:00+00:00,0.000,0.00,0.020,0.0,0.43,0.0,0.000,3.843,0.0,0.0,0.701


In [3]:
residential4_daily_data.head(3)

,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,energy_demand
utc_timestamp,,,,,,,,,,,
2015-10-14 00:00:00+00:00,0.009458,3.801208,0.344667,4.847500,6.491625,9.906250,0.004542,3.088417,5.377025,55.964250,8.733000
2015-10-15 00:00:00+00:00,0.473000,3.952000,0.509542,4.565792,5.277917,10.963958,0.006667,3.882417,1.886054,43.256808,8.840375
2015-10-16 00:00:00+00:00,0.468375,2.327250,0.498083,2.365208,10.344583,6.637250,0.004750,4.813750,1.658937,41.459929,13.924917


In [4]:
data_hourly = residential4_hourly_data[['energy_demand']].copy()
data_daily = residential4_daily_data[['energy_demand']].copy()

In [5]:
de_holidays = holidays.CountryHoliday(
    'DE',
    subdiv='BW',   # Baden-Württemberg
    years=range(2015, 2018)
)
holiday_dates = pd.to_datetime(list(de_holidays.keys()))
holiday_dates

DatetimeIndex(['2016-01-01', '2016-03-25', '2016-03-28', '2016-05-01',
               '2016-05-05', '2016-05-16', '2016-10-03', '2016-12-25',
               '2016-12-26', '2016-01-06', '2016-05-26', '2016-11-01',
               '2017-01-01', '2017-04-14', '2017-04-17', '2017-05-01',
               '2017-05-25', '2017-06-05', '2017-10-03', '2017-12-25',
               '2017-12-26', '2017-10-31', '2017-01-06', '2017-06-15',
               '2017-11-01', '2015-01-01', '2015-04-03', '2015-04-06',
               '2015-05-01', '2015-05-14', '2015-05-25', '2015-10-03',
               '2015-12-25', '2015-12-26', '2015-01-06', '2015-06-04',
               '2015-11-01'],
              dtype='datetime64[s]', freq=None)

In [6]:
data_hourly['is_holiday_or_weekend'] = data_hourly.index.to_series().apply(lambda x: x in holiday_dates or x.weekday() >= 5)
data_hourly.head(3)

,energy_demand,is_holiday_or_weekend
utc_timestamp,,
2015-10-13 17:00:00+00:00,1.248,False
2015-10-13 18:00:00+00:00,0.765,False
2015-10-13 19:00:00+00:00,0.701,False


In [7]:
data_daily['is_holiday_or_weekend'] = data_daily.index.to_series().apply(lambda x: x in holiday_dates or x.weekday() >= 5)
data_daily.head(3)  

,energy_demand,is_holiday_or_weekend
utc_timestamp,,
2015-10-14 00:00:00+00:00,8.733000,False
2015-10-15 00:00:00+00:00,8.840375,False
2015-10-16 00:00:00+00:00,13.924917,False


In [8]:
city = LocationInfo("Konstanz", "Germany")
data_hourly['daylight_flag'] = data_hourly.index.to_series().apply(lambda x: dawn(city.observer, date=x.date()) <= x <= dusk(city.observer, date=x.date()))
data_hourly.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag
utc_timestamp,,,
2015-10-13 17:00:00+00:00,1.248,False,True
2015-10-13 18:00:00+00:00,0.765,False,False
2015-10-13 19:00:00+00:00,0.701,False,False


In [9]:
def time_of_day(date): # has to be converted to the numerical format to be used in the model
    hour = date.hour
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'evening'
    
data_hourly['time_of_day'] = data_hourly.index.to_series().apply(time_of_day)
data_hourly.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day
utc_timestamp,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon
2015-10-13 18:00:00+00:00,0.765,False,False,evening
2015-10-13 19:00:00+00:00,0.701,False,False,evening


In [10]:
def get_season(date):  # has to be converted to the numerical format to be used in the model
    month = date.month
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'
data_hourly['season'] = data_hourly.index.to_series().apply(get_season)
data_hourly.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,False,False,evening,autumn


In [11]:
data_daily['season'] = data_daily.index.to_series().apply(get_season)
data_daily.head(3)

,energy_demand,is_holiday_or_weekend,season
utc_timestamp,,,
2015-10-14 00:00:00+00:00,8.733000,False,autumn
2015-10-15 00:00:00+00:00,8.840375,False,autumn
2015-10-16 00:00:00+00:00,13.924917,False,autumn


In [12]:
residential4_hourly_data = residential4_hourly_data.drop(columns=['energy_demand'])
data2 = data_hourly.copy()
residential4_hourly_merged_all = residential4_hourly_data.join(data2, how='left')
cols = ['energy_demand'] + [c for c in residential4_hourly_merged_all.columns if c != 'energy_demand']
residential4_hourly_merged_all = residential4_hourly_merged_all[cols]
residential4_hourly_merged_all.head()

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,,,,,,,,,,,
2015-10-13 17:00:00+00:00,1.248,0.002,0.57,0.018,0.0,0.470,0.0,0.002,5.224,0.0,0.0,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,0.000,0.00,0.026,0.0,0.530,0.0,0.000,4.540,0.0,0.0,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,0.000,0.00,0.020,0.0,0.430,0.0,0.000,3.843,0.0,0.0,False,False,evening,autumn
2015-10-13 20:00:00+00:00,0.573,0.000,0.00,0.017,0.0,0.350,0.0,0.000,3.196,0.0,0.0,False,False,evening,autumn
2015-10-13 21:00:00+00:00,0.662,0.000,0.00,0.029,0.0,0.436,0.0,0.000,2.757,0.0,0.0,False,False,evening,autumn


In [13]:
residential4_daily_data = residential4_daily_data.drop(columns=['energy_demand'])
data2 = data_daily.copy()
residential4_daily_merged_all = residential4_daily_data.join(data2, how='left')
cols = ['energy_demand'] + [c for c in residential4_daily_merged_all.columns if c != 'energy_demand']
residential4_daily_merged_all = residential4_daily_merged_all[cols]
residential4_daily_merged_all.head()

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,season
utc_timestamp,,,,,,,,,,,,,
2015-10-14 00:00:00+00:00,8.733000,0.009458,3.801208,0.344667,4.847500,6.491625,9.906250,0.004542,3.088417,5.377025,55.964250,False,autumn
2015-10-15 00:00:00+00:00,8.840375,0.473000,3.952000,0.509542,4.565792,5.277917,10.963958,0.006667,3.882417,1.886054,43.256808,False,autumn
2015-10-16 00:00:00+00:00,13.924917,0.468375,2.327250,0.498083,2.365208,10.344583,6.637250,0.004750,4.813750,1.658937,41.459929,False,autumn
2015-10-17 00:00:00+00:00,16.025792,0.009917,4.115250,0.495875,2.849958,11.119208,7.825333,0.004750,5.372500,16.392312,62.422004,True,autumn
2015-10-18 00:00:00+00:00,17.262167,0.581167,5.184708,0.492042,1.611875,10.978875,6.890042,0.007833,5.974083,16.178633,60.851613,True,autumn


In [14]:
print("Hourly data:")
print(f'\nShape: {residential4_hourly_merged_all.shape}')
print(f'\nNull values:\n{residential4_hourly_merged_all.isnull().sum()}')

Hourly data:

Shape: (12238, 15)

Null values:
energy_demand                   0
dishwasher                      0
ev                              0
freezer                         0
grid_export                     0
heat_pump                       0
pv                              0
washing_machine                 0
temperature                     0
radiation_direct_horizontal     0
radiation_diffuse_horizontal    0
is_holiday_or_weekend           0
daylight_flag                   0
time_of_day                     0
season                          0
dtype: int64


In [15]:
print("Daily data:")
print(f'\nShape: {residential4_daily_merged_all.shape}')
print(f'\nNull values:\n{residential4_daily_merged_all.isnull().sum()}')

Daily data:

Shape: (510, 13)

Null values:
energy_demand                   0
dishwasher                      0
ev                              0
freezer                         0
grid_export                     0
heat_pump                       0
pv                              0
washing_machine                 0
temperature                     0
radiation_direct_horizontal     0
radiation_diffuse_horizontal    0
is_holiday_or_weekend           0
season                          0
dtype: int64


In [16]:
# Split the data into training and testing sets based on the year
daily_train = residential4_daily_merged_all.loc[residential4_daily_merged_all.index.year != 2017]
daily_test = residential4_daily_merged_all.loc[residential4_daily_merged_all.index.year == 2017]
hourly_train = residential4_hourly_merged_all.loc[residential4_hourly_merged_all.index.year != 2017]
hourly_test = residential4_hourly_merged_all.loc[residential4_hourly_merged_all.index.year == 2017]


In [17]:
# save the full processed data
processed_data_dir = os.path.join(parent_dir, 'data/processed')
os.makedirs(processed_data_dir, exist_ok=True)
# save hourly data
processed_data_hourly_path = os.path.join(processed_data_dir, 'residential4_energy_demand_hourly.csv')
residential4_hourly_merged_all.to_csv(processed_data_hourly_path)
processed_data_hourly_train_path = os.path.join(processed_data_dir, 'residential4_energy_demand_hourly_train.csv')
hourly_train.to_csv(processed_data_hourly_train_path)
processed_data_hourly_test_path = os.path.join(processed_data_dir, 'residential4_energy_demand_hourly_test.csv')
hourly_test.to_csv(processed_data_hourly_test_path)
# save daily data
processed_data_daily_path = os.path.join(processed_data_dir, 'residential4_energy_demand_daily.csv')
residential4_daily_merged_all.to_csv(processed_data_daily_path)
processed_data_daily_train_path = os.path.join(processed_data_dir, 'residential4_energy_demand_daily_train.csv')
daily_train.to_csv(processed_data_daily_train_path)
processed_data_daily_test_path = os.path.join(processed_data_dir, 'residential4_energy_demand_daily_test.csv')
daily_test.to_csv(processed_data_daily_test_path)

In [18]:
# read newly saved hourly data to verify
verified_data = pd.read_csv(processed_data_hourly_path, index_col=0, parse_dates=True)
verified_data.head(3)

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,,,,,,,,,,,
2015-10-13 17:00:00+00:00,1.248,0.002,0.57,0.018,0.0,0.47,0.0,0.002,5.224,0.0,0.0,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,0.000,0.00,0.026,0.0,0.53,0.0,0.000,4.540,0.0,0.0,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,0.000,0.00,0.020,0.0,0.43,0.0,0.000,3.843,0.0,0.0,False,False,evening,autumn


In [19]:
# read newly saved daily data to verify
verified_data = pd.read_csv(processed_data_daily_path, index_col=0, parse_dates=True)
verified_data.head(3)   

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,season
utc_timestamp,,,,,,,,,,,,,
2015-10-14 00:00:00+00:00,8.733000,0.009458,3.801208,0.344667,4.847500,6.491625,9.906250,0.004542,3.088417,5.377025,55.964250,False,autumn
2015-10-15 00:00:00+00:00,8.840375,0.473000,3.952000,0.509542,4.565792,5.277917,10.963958,0.006667,3.882417,1.886054,43.256808,False,autumn
2015-10-16 00:00:00+00:00,13.924917,0.468375,2.327250,0.498083,2.365208,10.344583,6.637250,0.004750,4.813750,1.658937,41.459929,False,autumn
